# **Sección 1. Problema y datos**

---

### **1.1 ¿Qué queremos analizar y por qué?**

Cuando se habla de pobreza en Colombia, normalmente se resume la situación con una sola cifra: el porcentaje de hogares que se encuentran en condición de pobreza. Sin embargo, esa cifra no nos dice cómo son esos hogares ni cuáles son las dificultades específicas que enfrentan.

Esto nos lleva a plantear una pregunta más interesante:

> **¿Todos los hogares pobres presentan las mismas carencias o existen distintos perfiles de pobreza que requieren intervenciones diferentes?**

Para responder esta pregunta utilizaremos la información del **Índice de Pobreza Multidimensional (IPM)**, una medida que busca capturar diferentes aspectos del bienestar de los hogares más allá del ingreso. El IPM está construido a partir de **15 indicadores de privación**, agrupados en cinco dimensiones principales:

* **Educación:** logro educativo, analfabetismo, inasistencia escolar y rezago escolar.
* **Salud:** acceso a servicios de salud, aseguramiento, barreras de acceso y trabajo infantil.
* **Trabajo:** desempleo de larga duración y empleo informal.
* **Vivienda:** material de pisos, material de paredes y hacinamiento.
* **Servicios públicos:** acceso a acueducto y alcantarillado.

Cada hogar encuestado puede describirse mediante estas 15 variables, que indican si presenta o no cada una de las privaciones consideradas por el IPM.

Nuestro interés es analizar si, a partir de estas características, es posible identificar **grupos de hogares con patrones similares de privación**. Por ejemplo, algunos hogares podrían concentrar sus dificultades en aspectos relacionados con la vivienda y los servicios públicos, mientras que otros podrían presentar principalmente rezagos en educación o empleo.

Para explorar esta estructura utilizaremos técnicas de análisis multivariado y aprendizaje no supervisado:

* **PCA** para reducir la dimensionalidad de los datos y visualizar posibles patrones.
* **K-Means** y **clustering jerárquico** para identificar grupos de hogares con características similares.

De esta manera, buscaremos comprender un poco mejor la pobreza multidimensional en Colombia y describir los distintos perfiles que pueden existir detrás de una misma cifra.


### **1.2 Carga del entorno y dataset**
Antes de iniciar el análisis, se cargan las librerías necesarias para el procesamiento de datos, la visualización de resultados y la aplicación de las técnicas de reducción de dimensionalidad y agrupamiento que se utilizarán más adelante.

También, se importa la base de datos correspondiente al **Índice de Pobreza Multidimensional (IPM)**, que fue tomada de la página de microdatos del [DANE](https://microdatos.dane.gov.co/index.php/catalog/903/get-microdata). Este conjunto de datos contiene la información descrita en la sección anterior


In [2]:
# Librerías
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# Estilo visual
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [6]:
# Carga del dataset desde Github con Pandas
url = "https://raw.githubusercontent.com/JohanAv1018/Proyecto-Final---Diplomado/main/IPM2025.csv"
ipm = pd.read_csv(url)


### **1.3 Exploración y limpieza de variables**
Una vez cargada la información, se realiza una revisión inicial de la estructura de los datos con el fin de verificar que las variables hayan sido importadas correctamente e identificar posibles problemas de calidad, tales como valores faltantes o inconsistencias en los registros y poder realizar la limpieza de ser necesaria.


#### **1.3.1 Nuevo dataset para análisis**

Vamos a crear un dataset nuevo a partir del ya existente, que contenga las variables del IPM, junto con las variables `DEPARTAMENTO`, `POBRE` e `IPM`, que son las variables que nos interesa usar para el análisis.

In [9]:
variables_ipm = [
    'logro_educativo',
    'analfabetismo',
    'inasistencia_escolar',
    'rezago_escolar',
    'atencion_integral',
    'trabajo_infantil',
    'aseguramiento_salud',
    'barreras_acceso_salud',
    'desempleo_larga_duracion',
    'empleo_formal',
    'acueducto',
    'alcantarillado',
    'pisos',
    'paredes',
    'hacinamiento'
]

variables_contexto = [
    'IPM',
    'POBRE',
    'DEPARTAMENTO'
]

ipm_analisis = ipm[variables_ipm + variables_contexto].copy()

#### **1.3.2 Dimensiones y primer vistazo**

In [15]:
# Dimensiones del conjunto de datos
print("Dimensiones:", ipm.shape)

# Información general
ipm_analisis.info()

Dimensiones: (87060, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87060 entries, 0 to 87059
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   logro_educativo           87060 non-null  int64  
 1   analfabetismo             87060 non-null  int64  
 2   inasistencia_escolar      87060 non-null  int64  
 3   rezago_escolar            87060 non-null  int64  
 4   atencion_integral         87060 non-null  int64  
 5   trabajo_infantil          87060 non-null  int64  
 6   aseguramiento_salud       87060 non-null  int64  
 7   barreras_acceso_salud     87060 non-null  int64  
 8   desempleo_larga_duracion  87060 non-null  int64  
 9   empleo_formal             87060 non-null  int64  
 10  acueducto                 87060 non-null  int64  
 11  alcantarillado            87060 non-null  int64  
 12  pisos                     87060 non-null  int64  
 13  paredes                   87060 non-

#### **1.3.3 Valores faltantes**

In [11]:
# Valores faltantes
ipm_analisis.isna().sum().sort_values(ascending=False)

,0
logro_educativo,0
analfabetismo,0
inasistencia_escolar,0
rezago_escolar,0
atencion_integral,0
trabajo_infantil,0
aseguramiento_salud,0
barreras_acceso_salud,0
desempleo_larga_duracion,0
empleo_formal,0


Podemos observar que no se encuentran valores faltantes en ninguna de las variables que vamos a analizar.

#### **1.3.4 Registros duplicados**

In [13]:
# Registros duplicados con dataset base
duplicados = ipm.duplicated().sum()
print(f"Registros duplicados: {duplicados}")

Registros duplicados: 0


Al hacer la verificación de registros duplicados con el dataset original, vemos que no tenemos ninguno, pues ningún hogar es exactamente igual a otro.

Como no tenemos ni registros duplicados, ni valores faltantes, podemos comenzar a utilizar el dataset para el análisis planteado inicialmente.